%md
# Eval 01 · Rule baseline
Measures how well the label-free alert rules find laundering. Reads labels from silver and aml_eval. Nothing here feeds back into the rules.

In [0]:
from pyspark.sql import functions as F

GOLD = "workspace.aml_gold"
SILVER = "workspace.aml_silver"
EVAL = "workspace.aml_eval"

alerts = spark.table(f"{GOLD}.alerts")
all_accounts = spark.table(f"{GOLD}.account_features").select("account_id", "is_hub")
laundering = spark.table(f"{SILVER}.transactions").filter("is_laundering = 1")

involved = (
    laundering.select(F.col("from_account_id").alias("account_id"))
    .union(laundering.select(F.col("to_account_id").alias("account_id")))
    .distinct()
    .withColumn("is_involved", F.lit(True))
)

print(f"Accounts involved in laundering: {involved.count():,}")

In [0]:
scored = (
    all_accounts
    .join(alerts.select("account_id", "priority", "triggered_rules"), on="account_id", how="left")
    .join(involved, on="account_id", how="left")
    .fillna(False, subset=["is_involved"])
    .withColumn("is_alerted", F.col("priority").isNotNull())
)

m = scored.agg(
    F.count("*").alias("accounts"),
    F.sum(F.col("is_involved").cast("int")).alias("involved"),
    F.sum(F.col("is_alerted").cast("int")).alias("alerted"),
    F.sum((F.col("is_alerted") & F.col("is_involved")).cast("int")).alias("true_positives"),
).first()

precision = m["true_positives"] / m["alerted"]
recall = m["true_positives"] / m["involved"]
base_rate = m["involved"] / m["accounts"]

print(f"Alerts:          {m['alerted']:,}")
print(f"True positives:  {m['true_positives']:,}")
print(f"Precision:       {precision:.1%}")
print(f"Recall:          {recall:.1%}")
print(f"Base rate:       {base_rate:.2%}")
print(f"Lift:            {precision / base_rate:.1f}x")

In [0]:
alerted = scored.filter("is_alerted")

display(
    alerted.groupBy("priority")
    .agg(
        F.count("*").alias("alerts"),
        F.avg(F.col("is_involved").cast("int")).alias("precision"),
    )
    .orderBy("priority")
)

display(
    alerted
    .withColumn("rule", F.explode("triggered_rules"))
    .groupBy("rule")
    .agg(
        F.count("*").alias("alerts_with_rule"),
        F.avg(F.col("is_involved").cast("int")).alias("precision"),
    )
    .orderBy(F.desc("precision"))
)

In [0]:
patterns = spark.table(f"{EVAL}.pattern_labels").select("txn_id", "pattern_type")
alerted_ids = alerts.select("account_id")

labelled = (
    laundering.select("txn_id", "from_account_id", "to_account_id")
    .join(patterns, on="txn_id", how="left")
    .fillna("UNCLASSIFIED", subset=["pattern_type"])
    .join(alerted_ids.withColumnRenamed("account_id", "from_account_id").withColumn("from_alerted", F.lit(True)), on="from_account_id", how="left")
    .join(alerted_ids.withColumnRenamed("account_id", "to_account_id").withColumn("to_alerted", F.lit(True)), on="to_account_id", how="left")
    .fillna(False, subset=["from_alerted", "to_alerted"])
    .withColumn("caught", F.col("from_alerted") | F.col("to_alerted"))
)

print(f"Laundering txns touching an alerted account: {labelled.filter('caught').count():,} of {labelled.count():,}")

display(
    labelled.groupBy("pattern_type")
    .agg(
        F.count("*").alias("txns"),
        F.avg(F.col("caught").cast("int")).alias("recall"),
    )
    .orderBy(F.desc("recall"))
)